# τ ablation: TAAD + audio with τ fixed at 0

Same training as `curr_model_40` (resume from the video-only `curr_model_20`, epochs 21 to 40), except the audio delay τ is frozen at 0.

**Before running:** GPU on, Internet on, secrets `HF_TOKEN` and `FOOTPASS_ZIP_PASSWORD` added, and your dataset (v3 zip + `checkpointscurr_model_20.pt`) attached.
Every session: **Save Version → Save & Run All**. The notebook resumes automatically from the newest checkpoint of the previous session once you attach that output.

In [ ]:
# CELL 1: GPU check + copy the code
import os, glob, shutil, subprocess, zipfile, torch

assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> GPU"
print("GPU:", torch.cuda.get_device_name(0))

CODE = "/kaggle/working/FOOTPASS"

code_src = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "train_TAAD_Baseline.py" in files and "models" in dirs:
        code_src = root
        break

if code_src is None:  # Kaggle kept the zip zipped
    zips = glob.glob("/kaggle/input/**/FOOTPASS-main-audiofusion-v3.zip", recursive=True)
    assert zips, "Code not found: attach the dataset with FOOTPASS-main-audiofusion-v3.zip"
    zipfile.ZipFile(zips[0]).extractall("/kaggle/temp/code")
    code_src = "/kaggle/temp/code/FOOTPASS-main"

shutil.copytree(code_src, CODE, dirs_exist_ok=True)
assert "--fixed_tau" in open(f"{CODE}/train_TAAD_Baseline.py").read(), "Old code: upload the v3 zip"
print("Code copied from", code_src)

!pip install -q decord albumentations fvcore iopath

In [ ]:
# CELL 2: download the FOOTPASS data from Hugging Face (about 15-30 min, every session)
import os, time
# Fast Xet downloads, but with fewer parallel connections than the default to avoid HTTP 429.
# Must be set before huggingface_hub is imported -> restart the kernel before running this cell.
os.environ["HF_XET_NUM_CONCURRENT_RANGE_GETS"] = "8"
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download

def download_with_retry(filename, tries=6):
    for attempt in range(1, tries + 1):
        try:
            return hf_hub_download(repo_id="SoccerNet/SN-PCBAS-2026", filename=filename, repo_type="dataset",
                                   local_dir="/kaggle/temp/dl", token=HF_TOKEN)
        except Exception as e:
            if attempt == tries:
                raise
            wait = 60 * attempt
            print(f"  download failed ({str(e)[:120]}...) -> retry {attempt}/{tries - 1} in {wait} s")
            time.sleep(wait)

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
ZIP_PW = secrets.get_secret("FOOTPASS_ZIP_PASSWORD")

DATA_ROOT = "/kaggle/temp/FOOTPASS"      # not saved between sessions, but has the disk space
os.makedirs(f"{DATA_ROOT}/data", exist_ok=True)
os.makedirs(f"{DATA_ROOT}/videos", exist_ok=True)

def n_videos():
    return len(glob.glob(f"{DATA_ROOT}/videos/*.mp4"))

downloads = [
    ("tactical_data_TRAIN.zip", "data",   lambda: os.path.exists(f"{DATA_ROOT}/data/train_tactical_data.h5")),
    ("tactical_data_VAL.zip",   "data",   lambda: os.path.exists(f"{DATA_ROOT}/data/val_tactical_data.h5")),
    ("videos_352x640_VAL.zip",  "videos", lambda: n_videos() >= 3),
    ("videos_352x640_TRAIN.zip","videos", lambda: n_videos() >= 51),
]

for filename, sub, done in downloads:
    if done():
        print("already there:", filename)
        continue
    print("downloading", filename, "| free disk GB:", shutil.disk_usage("/kaggle/temp").free // 2**30)
    path = download_with_retry(filename)
    subprocess.run(["7z", "x", f"-p{ZIP_PW}", f"-o{DATA_ROOT}/{sub}", path, "-y"],
                   check=True, stdout=subprocess.DEVNULL)
    os.remove(path)

# the zips may contain sub-folders: move files to where the code expects them
for f in glob.glob(f"{DATA_ROOT}/videos/**/*.mp4", recursive=True):
    if os.path.dirname(f) != f"{DATA_ROOT}/videos":
        shutil.move(f, f"{DATA_ROOT}/videos/{os.path.basename(f)}")
for f in glob.glob(f"{DATA_ROOT}/data/**/*_tactical_data.h5", recursive=True):
    if os.path.dirname(f) != f"{DATA_ROOT}/data":
        shutil.move(f, f"{DATA_ROOT}/data/{os.path.basename(f)}")

for f in ["TAAD_sample_list.json", "audio_features.h5"]:
    shutil.copy(f"{CODE}/data/{f}", f"{DATA_ROOT}/data/{f}")

print("videos:", n_videos(), "| data files:", sorted(os.listdir(f"{DATA_ROOT}/data")))

In [ ]:
# CELL 3: choose the checkpoint to start from (never curr_model_40 of the learned-tau run)
RUN = "/kaggle/working/runs_fixed_tau"

def epoch_of(p):
    return int(os.path.basename(p).rsplit("_", 1)[-1].split(".")[0])

previous = glob.glob("/kaggle/input/**/runs_fixed_tau/checkpoints/curr_model_*.pt", recursive=True)
if previous:
    RESUME = max(previous, key=epoch_of)
    print("Continuing the ablation from a previous session:", RESUME)
else:
    start = glob.glob("/kaggle/input/**/*curr_model_20.pt", recursive=True)
    assert start, "Attach the dataset containing checkpointscurr_model_20.pt"
    RESUME = start[0]
    print("First session: starting from the video-only checkpoint", RESUME)

print("Training will start at epoch", epoch_of(RESUME) + 1, "and stop after epoch 40")
assert epoch_of(RESUME) < 40, "Already finished: go to CELL 5"

sd = torch.load(RESUME, map_location="cpu", weights_only=False)["model_state_dict"]
if "audio_head.tau" in sd:
    assert float(sd["audio_head.tau"].abs().max()) == 0.0, "Wrong checkpoint: it has a learned tau"
print("Checkpoint OK (no learned tau)")
del sd


In [ ]:
# CELL 4: train with tau fixed at 0
# Check the first lines of the log: "[INFO] Ablation: audio tau fixed at 0" and "Resuming training from epoch ..."
# batch 2 x accum 24 = 48 clips per update, same recipe as curr_model_20 (about 2.3 GB of GPU memory per clip)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
%cd /kaggle/working/FOOTPASS
!python train_TAAD_Baseline.py \
    --data_root {DATA_ROOT} \
    --run_path {RUN} \
    --batch_size 2 --accum_steps 24 --num_workers 2 \
    --epochs 40 \
    --resume_path {RESUME} \
    --fixed_tau

### After epoch 40 is finished
Run CELL 5 (in a session where CELL 1-3 have run). It makes the val predictions to send back for the comparison with learned τ.

In [ ]:
# CELL 5: val predictions with the finished ablation model (about 5 h)
final = sorted(glob.glob(f"{RUN}/checkpoints/curr_model_40.pt") +
               glob.glob("/kaggle/input/**/runs_fixed_tau/checkpoints/curr_model_40.pt", recursive=True))
assert final, "curr_model_40.pt of the ablation not found yet: training not finished"
import torch as _t
assert float(_t.load(final[0], map_location="cpu", weights_only=False)["model_state_dict"]["audio_head.tau"].abs().max()) == 0.0, "tau is not 0!"
print("Using", final[0], "(tau confirmed 0)")

%cd /kaggle/working/FOOTPASS
!python run_TAAD_on_matches.py \
    --model_checkpoint {final[0]} \
    --data_root {DATA_ROOT} \
    --save_path /kaggle/working/val_preds_fixed_tau \
    --set_to_run val --start_index 0 --end_index 6